# **Dataset Exploration — Kenya Retail Price Estimates (RTFP)**
### ICS 3202 Semester Project | Deliverable 1
**Application:** Price Check Kenya — a supermarket price comparison and smart shopping app for Kenyan consumers.


## 1. Candidate Open-Source Datasets

Datasets considered for the ML engine, and the variable each could supply as the expected **output** of the final application:

| # | Dataset | Source | Candidate Output Variable |
|---|---------|--------|----------------------------|
| 1 | **Kenya — Monthly Food Price Estimates by Product and Market (RTFP)** | World Bank Microdata Library | `inflation_<commodity>` → derived **`price_direction`** (Increase / Decrease / Stable) per commodity, already provided as a percent-change field in the raw data |
| 2 | Kenya — National Average Retail Prices of Selected Commodities | KNBS via Humanitarian Data Exchange (HDX) | Average national retail price (KSh) → price direction, if the series has enough of a time dimension to derive it |
| 3 | Supermarket Price Dataset | Kaggle (anapedralpez) | `price` per product |
| 4 | Detailed Products Dataset | Kaggle (sujaykapadnis) | `price` per product/category |
| 5 | Supermarket Sales Dataset | Kaggle (faresashraf1001) | `unit_price` / `total` per transaction |


## 2. Selected Dataset

**Kenya — Monthly Food Price Estimates by Product and Market (RTFP)**, World Bank Development Economics Data Group.
- Source: https://microdata.worldbank.org/index.php/catalog/6167
- File: `KEN_RTFP_mkt_2007_2026-08-24.csv` (version 2026-08-24)
- Coverage: 233 markets across Kenya, monthly, Jan 2007 – Aug 2026
- The file is in **wide format**: one row per market per month. Instead of a single generic `price` column, each commodity tracked (`beans`, `livestock_goat_s_fao`, `maize_fao`) has its own set of columns — the headline price estimate, plus `o_`/`h_`/`l_`/`c_` (open/high/low/close) estimates, an `inflation_<commodity>` percent-change field, and a `trust_<commodity>` confidence score. A `food_price_index` block aggregates across commodities the same way.
- Key identifying/location fields: `price_date`, `year`, `month`, `adm1_name` (region), `adm2_name`, `mkt_name` (market), `lat`/`lon`.

**Why this dataset:** it is real, Kenya-specific retail price data (not synthetic), and its `inflation_<commodity>` fields already give a ready-made **`price_direction`** signal (increase/decrease/stable) once thresholded — no manual differencing required. Its time-series structure (repeated monthly observations per market) supports the app's **price direction and price trend analysis** feature.

> Citation: Andrée, B. P. J. (2021). *Monthly food price estimates by product and market* (Kenya, Version 2026-08-24). KEN_2021_RTFP_v02_M. Washington, DC: World Bank Microdata Library.


## 3. Load the Data

The file requires a free World Bank Microdata Library account to download (no API key). Download `KEN_RTFP_mkt_2007_2026-08-24.csv` from the link above, then upload it below (or place it in your Google Drive and adjust the path).

In [2]:
# Option A: upload the CSV directly in Colab
from google.colab import files
import pandas as pd

uploaded = files.upload()  # select KEN_RTFP_mkt_2007_2026-08-24.csv when prompted
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print("Columns available:", list(df.columns))
df.head()

Saving KEN_RTFP_mkt_2007_2026-08-24.csv to KEN_RTFP_mkt_2007_2026-08-24 (1).csv
Columns available: ['ISO3', 'country', 'adm1_name', 'adm2_name', 'mkt_name', 'lat', 'lon', 'geo_id', 'price_date', 'year', 'month', 'currency', 'components', 'start_dense_data', 'last_survey_point', 'data_coverage', 'data_coverage_recent', 'index_confidence_score', 'spatially_interpolated', 'beans', 'livestock_goat_s_fao', 'maize_fao', 'o_beans', 'h_beans', 'l_beans', 'c_beans', 'inflation_beans', 'trust_beans', 'o_livestock_goat_s_fao', 'h_livestock_goat_s_fao', 'l_livestock_goat_s_fao', 'c_livestock_goat_s_fao', 'inflation_livestock_goat_s_fao', 'trust_livestock_goat_s_fao', 'o_maize_fao', 'h_maize_fao', 'l_maize_fao', 'c_maize_fao', 'inflation_maize_fao', 'trust_maize_fao', 'o_food_price_index', 'h_food_price_index', 'l_food_price_index', 'c_food_price_index', 'inflation_food_price_index', 'trust_food_price_index']


,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,price_date,year,...,l_maize_fao,c_maize_fao,inflation_maize_fao,trust_maize_fao,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
0,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-01-01,2007,...,21.29,22.23,NaN,9.7,0.50,0.52,0.47,0.50,NaN,9.0
1,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-02-01,2007,...,21.44,21.67,NaN,9.7,0.50,0.52,0.47,0.49,NaN,9.0
2,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-03-01,2007,...,19.51,19.51,NaN,9.7,0.48,0.51,0.46,0.49,NaN,9.0
3,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-04-01,2007,...,18.74,18.77,NaN,9.7,0.49,0.52,0.47,0.49,NaN,9.0
4,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-05-01,2007,...,16.90,16.90,NaN,9.7,0.48,0.51,0.46,0.46,NaN,9.0


# **Data Preprocessing and Exploratory Data Analysis (EDA)**

### a. How many rows and columns are contained in the dataset?

In [3]:
n_rows, n_cols = df.shape
print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")

Rows: 55224
Columns: 46


### b. What datatypes are contained in the dataset?

In [4]:
df.dtypes

,0
ISO3,object
country,object
adm1_name,object
adm2_name,object
mkt_name,object
lat,float64
lon,float64
geo_id,object
price_date,object
year,int64


### c. Is the dataset complete? i.e. no missing values?

In [5]:
missing_per_column = df.isnull().sum()
total_missing = missing_per_column.sum()

print(missing_per_column[missing_per_column > 0])
print(f"\nTotal missing values: {total_missing}")
print("Dataset is COMPLETE (no missing values)." if total_missing == 0
      else "Dataset is NOT complete — missing values found above.")

lat                                 236
lon                                 236
beans                             53429
livestock_goat_s_fao              52671
maize_fao                         52227
inflation_beans                    2808
inflation_livestock_goat_s_fao     2808
inflation_maize_fao                2808
inflation_food_price_index         2808
dtype: int64

Total missing values: 170031
Dataset is NOT complete — missing values found above.


### d. Slice out the first 15 rows and last 20 rows, merge into `df_sample`

In [6]:
df_sample = pd.concat([df.head(15), df.tail(20)], ignore_index=True)
df_sample

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,price_date,year,...,l_maize_fao,c_maize_fao,inflation_maize_fao,trust_maize_fao,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
0,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-01-01,2007,...,21.29,22.23,NaN,9.7,0.50,0.52,0.47,0.50,NaN,9.0
1,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-02-01,2007,...,21.44,21.67,NaN,9.7,0.50,0.52,0.47,0.49,NaN,9.0
2,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-03-01,2007,...,19.51,19.51,NaN,9.7,0.48,0.51,0.46,0.49,NaN,9.0
3,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-04-01,2007,...,18.74,18.77,NaN,9.7,0.49,0.52,0.47,0.49,NaN,9.0
4,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-05-01,2007,...,16.90,16.90,NaN,9.7,0.48,0.51,0.46,0.46,NaN,9.0
5,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-06-01,2007,...,16.13,17.18,NaN,9.7,0.46,0.48,0.44,0.45,NaN,9.0
6,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-07-01,2007,...,16.46,17.29,NaN,9.7,0.45,0.47,0.43,0.46,NaN,9.0
7,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-08-01,2007,...,14.66,14.66,NaN,9.7,0.46,0.49,0.44,0.49,NaN,9.0
8,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-09-01,2007,...,13.93,14.95,NaN,9.7,0.50,0.52,0.48,0.48,NaN,9.0
9,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-10-01,2007,...,14.21,18.49,NaN,9.7,0.48,0.50,0.45,0.45,NaN,9.0


## 4. Mapping the Dataset to the Price Check Kenya UI

The app mockup defines specific screens the ML output needs to power. Checking each against what RTFP actually contains:

| UI Screen | What it needs | Does RTFP support it? |
|---|---|---|
| Home — "Today's Price Highlights" (Naivas KSh 145 vs Carrefour KSh 139 vs Quickmart KSh 150) | Per-retailer, per-store prices for the same product | **No.** RTFP has one price series per commodity (`beans`, `maize_fao`, etc.) per `mkt_name` (a geographic market, e.g. "Nairobi"), not per retail chain. There is no `Naivas` / `Carrefour` / `Quickmart` field. |
| Product Search / Price Comparison screens | Branded product catalog (Brookside vs Fresha vs Molo milk) with per-brand prices | **No.** Commodities are tracked at the generic staple level (beans, maize, livestock), not by brand/SKU. |
| Price Analysis screen ("price decreased 7% this month", line chart, current vs previous average) | Time series of price per commodity per location | **Yes.** `price_date`, `adm1_name`, `mkt_name`, and the per-commodity price columns give exactly this — RTFP's core strength. |
| Price Trends screen ("Sugar ↓ 4%", "Cooking oil ↑ 6%", AI market insight) | Month-over-month or period-over-period % change per commodity | **Yes**, and largely pre-computed — the `inflation_<commodity>` columns already give this directly. |
| Budget / Basket / Smart Recommendations screens | Multiple SKUs, quantities, and a live basket total against real retailer prices | **No.** These need a live product-price feed, not RTFP. |

**Takeaway:** RTFP alone can power the *Price Trends* and *Price Analysis* screens (price direction / percent change over time, by commodity and region). It cannot power the *retailer-level price comparison* shown on Home, Search, and the Compare screen — that needs a separate current-price source (e.g. the Kaggle supermarket datasets from Section 1, or manually collected retailer prices), which is why the project architecture should treat these as two distinct data sources feeding two distinct features, not one dataset doing both jobs.

### e. Deriving the Price Trends output (`price_direction`) from RTFP

The raw file already provides an `inflation_<commodity>` percent-change column for each tracked commodity (`beans`, `livestock_goat_s_fao`, `maize_fao`) plus the aggregate `food_price_index`. So the required Price Trends output — a direction label (Increase / Decrease / Stable) — can be derived by thresholding that existing field directly, without manually differencing consecutive price observations.

In [ ]:
df['price_date'] = pd.to_datetime(df['price_date'])

commodities = ['beans', 'livestock_goat_s_fao', 'maize_fao', 'food_price_index']

# threshold instead of exact zero, since inflation values fluctuate by small amounts
STABLE_THRESHOLD = 1.0  # percent

def classify(pct):
    if pd.isna(pct):
        return None
    if pct > STABLE_THRESHOLD:
        return 'Increase'
    if pct < -STABLE_THRESHOLD:
        return 'Decrease'
    return 'Stable'

trend_cols = ['price_date', 'mkt_name']
df_trend = df[trend_cols + [f'inflation_{c}' for c in commodities]].copy()

for c in commodities:
    df_trend[f'price_direction_{c}'] = df_trend[f'inflation_{c}'].apply(classify)

df_trend.sort_values('price_date').head(10)